# Homework-3 

## Use the Titanic Dataset to Practice Data Wrangling Tasks
### Part 1: Import libaries , load the data, inspect the data, then get the name of the variables and create a small markdown table  with variable names and types (e.g numerical or categorical ordinal, categorical nominal etc.)

### Part 2: 
    - Impute missing values for "Embarked" with the mode (most frequent value)
    - Impute missing value for Age using two following strategies
        - A) Simple mean imputation (fill with the overal mean age)
        - B) Group-wise median imputation: for each combination of Sex and Pclass compute the median age. Impute missing age using the median age of passenger's sex and Pclass group

### Part 3: Create a FamilySize variable (FamilySize = SibSp + Parch +1) and compute:
    - Minimum, Maximum, and mean FamilySize
    - How many passengers were traveling alone
    
### Part 4:  Create a filtered dataframe of all passengers:
    - Who are female
    - Who are male 
    - Who are under 16 years old   
    - Compute the survival rate for the above filtered data
    - Compute survival rate by Sex and Pclass (use groupby and mean) and present the result in a markedown table (rows: sex, columns: Pclass) or using print formating.
    - Interpret your result in 2-3 sentences
     
    
### Part 5: Compute the average fare for each Pclass (Fare values are per tickets not per person (there coud be more than 1 person on a ticket) so make sure to account for that. 

In [1]:
import pandas as pd

# Load titanic data file
df = pd.read_csv('titanic.csv')

# Inspect the loaded data
df.head()



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


| Variable Name | Description | Data Type Category |
| :--- | :--- | :--- |
| **PassengerId** | Unique ID for each passenger | Numerical (Discrete) / Identifier |
| **Survived** | Survival status (0 = No, 1 = Yes) | Categorical (Nominal) |
| **Pclass** | Passenger class (1 = 1st, 2 = 2nd, 3 = 3rd) | Categorical (Ordinal) |
| **Name** | Full passenger name | Categorical (Nominal) |
| **Sex** | Gender of the passenger | Categorical (Nominal) |
| **Age** | Age in years | Numerical (Continuous) |
| **SibSp** | Number of siblings/spouses aboard | Numerical (Discrete) |
| **Parch** | Number of parents/children aboard | Numerical (Discrete) |
| **Ticket** | Ticket number | Categorical (Nominal) |
| **Fare** | Passenger fare paid | Numerical (Continuous) |
| **Cabin** | Cabin number | Categorical (Nominal) |
| **Embarked** | Port of Embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) | Categorical (Nominal) |

In [4]:
# 1. Impute missing values for "Embarked" with the mode
embarked_mode = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(embarked_mode)

# 2A. Simple mean imputation (filling with overall mean age)
df_mean_imputed = df.copy()
mean_age = df_mean_imputed['Age'].mean()
df_mean_imputed['Age'] = df_mean_imputed['Age'].fillna(mean_age)

# 2B. Group-wise median imputation (by Sex and Pclass)
df_median_imputed = df.copy()
df_median_imputed['Age'] = df_median_imputed.groupby(['Sex', 'Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

# Using df_median_imputed for subsequent parts
df = df_median_imputed

In [3]:
# Create FamilySize variable
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Calculate Min, Max, Mean FamilySize
min_family = df['FamilySize'].min()
max_family = df['FamilySize'].max()
mean_family = df['FamilySize'].mean()

print(f"Minimum Family Size: {min_family}")
print(f"Maximum Family Size: {max_family}")
print(f"Mean Family Size: {mean_family:.2f}")

# Count passengers traveling alone (FamilySize == 1)
traveling_alone = (df['FamilySize'] == 1).sum()
print(f"Number of passengers traveling alone: {traveling_alone}")

Minimum Family Size: 1
Maximum Family Size: 11
Mean Family Size: 1.90
Number of passengers traveling alone: 537


In [5]:
# Create filtered DataFrames
df_female = df[df['Sex'] == 'female']
df_male = df[df['Sex'] == 'male']
df_under_16 = df[df['Age'] < 16]

# Compute survival rates
rate_female = df_female['Survived'].mean()
rate_male = df_male['Survived'].mean()
rate_under_16 = df_under_16['Survived'].mean()

print(f"Survival Rate - Females: {rate_female:.2%}")
print(f"Survival Rate - Males: {rate_male:.2%}")
print(f"Survival Rate - Under 16: {rate_under_16:.2%}\n")

# Compute survival rate by Sex and Pclass
survival_pivot = df.pivot_table(index='Sex', columns='Pclass', values='Survived', aggfunc='mean')
print("Survival Rate by Sex and Pclass:")
print(survival_pivot)

Survival Rate - Females: 74.20%
Survival Rate - Males: 18.89%
Survival Rate - Under 16: 59.04%

Survival Rate by Sex and Pclass:
Pclass         1         2         3
Sex                                 
female  0.968085  0.921053  0.500000
male    0.368852  0.157407  0.135447


**Interpretation:**
Female passengers had a significantly higher overall survival rate (74.20%) compared to male passengers (18.89%), reflecting a "women and children first" evacuation protocol. Children under 16 also had a higher survival rate (59.04%) than the overall average. Furthermore, passenger class played a major role, with 1st class passengers having the highest survival rates across both genders while 3rd class passengers had the lowest.

In [6]:
# Deduplicate by Ticket to get one record per ticket
unique_tickets = df.drop_duplicates(subset=['Ticket'])

# Compute the average fare per ticket for each Pclass
avg_fare_per_ticket = unique_tickets.groupby('Pclass')['Fare'].mean()

print("Average Fare Per Ticket by Pclass:")
print(avg_fare_per_ticket.round(2))

Average Fare Per Ticket by Pclass:
Pclass
1    64.14
2    17.51
3    10.08
Name: Fare, dtype: float64
